# 📊 Clase 3 — Tu primer dataset

Hoy vas a aprender a **cargar, explorar y entender un dataset** usando pandas.
Es la habilidad más fundamental en ciencia de datos: antes de modelar, hay que mirar los datos.

### ¿Qué vamos a cubrir?

| Sección | Concepto |
|---|---|
| 1 | Setup: instalar pandas y matplotlib |
| 2 | ¿Qué son los datos tabulares? |
| 3 | Generar un dataset sintético |
| 4 | Cargar y mirar los datos con pandas |
| 5 | Las 3 preguntas obligatorias |
| 6 | Estadísticas descriptivas |
| 7 | Visualización de distribuciones |
| 8 | Ejercicio evaluable |

> **Regla de oro:** nunca entrenes un modelo sin antes MIRAR los datos.

In [ ]:
# ── 1. Setup ────────────────────────────────────────────────
# pip install pandas matplotlib

import pandas as pd
import matplotlib.pyplot as plt
import random

random.seed(42)

print(f"✅ pandas     : {pd.__version__}")
print(f"✅ matplotlib : {plt.matplotlib.__version__}")
print("\n📍 Listo para explorar datos.")

---
## 2. ¿Qué son los datos tabulares?

Los datos tabulares son información organizada en **filas y columnas**, como una planilla de Excel:

| Término | Significado | Analogía |
|---|---|---|
| **Fila** | Un ejemplo / registro / observación | Un estudiante |
| **Columna** | Una variable / atributo / feature | Edad, nota, ciudad |
| **Valor** | El dato concreto en una celda | 22, 8.5, "Buenos Aires" |
| **DataFrame** | La tabla completa en pandas | La planilla completa |

### Tipos de datos comunes

| Tipo | Ejemplos | En pandas |
|---|---|---|
| **Numérico continuo** | Edad, salario, temperatura | `float64` |
| **Numérico entero** | Cantidad hijos, pisos | `int64` |
| **Categórico** | Ciudad, carrera, género | `object` |
| **Booleano** | ¿Aprobó? ¿Trabaja? | `bool` |
| **Fecha** | Fecha nacimiento, inscripción | `datetime64` |

> 💡 **En este curso usamos datos tabulares** que es lo más común en ML clásico. Más adelante veremos datos de texto para NLP.

---
## 3. Generar un dataset sintético

Para aprender sin depender de internet, vamos a **generar nuestros propios datos**.
Simulamos un dataset de **200 estudiantes** de una universidad ficticia.

| Columna | Tipo | Descripción |
|---|---|---|
| `nombre` | texto | Nombre del estudiante |
| `edad` | int | Edad (18-35) |
| `carrera` | categórica | Ingeniería, Sistemas, Diseño, Economía |
| `horas_estudio` | float | Horas de estudio semanal |
| `nota_parcial` | float | Nota del primer parcial (0-10) |
| `asistencia_pct` | float | Porcentaje de asistencia (0-100) |
| `trabaja` | bool | ¿Trabaja además de estudiar? |
| `nota_final` | float | Nota final (lo que queremos predecir) |

In [ ]:
# Generador de dataset sintético reproducible

random.seed(42)

nombres_pool = [
    "Ana", "Lucía", "Martín", "Sofía", "Mateo", "Valentina", "Santiago",
    "Isabella", "Sebastián", "Camila", "Nicolás", "Emilia", "Daniel",
    "Renata", "Tomás", "Paula", "Alejandro", "Florencia", "Lucas",
    "Catalina", "Diego", "Julieta", "Pablo", "Mía", "Gabriel",
]

apellidos_pool = [
    "García", "Martínez", "López", "González", "Rodríguez", "Pérez",
    "Sánchez", "Ramírez", "Torres", "Flores", "Rivera", "Gómez",
    "Díaz", "Cruz", "Morales", "Reyes", "Ortiz", "Herrera",
]

carreras = ["Ingeniería", "Sistemas", "Diseño", "Economía"]

N = 200  # cantidad de estudiantes

datos = []
for i in range(N):
    nombre = f"{random.choice(nombres_pool)} {random.choice(apellidos_pool)}"
    edad = random.randint(18, 35)
    carrera = random.choice(carreras)
    trabaja = random.random() < 0.4  # 40% trabaja
    
    # Horas de estudio: los que trabajan estudian menos
    base_horas = random.gauss(12, 5) if not trabaja else random.gauss(7, 4)
    horas_estudio = round(max(0, min(30, base_horas)), 1)
    
    # Asistencia: correlaciona con horas de estudio
    asistencia = round(
        min(
            100, 
            max(
                0, 
                horas_estudio * 4.5 + random.gauss(20, 15))
            ), 1)
    
    # Nota parcial: correlaciona con estudio y asistencia
    nota_parcial = round(min(10, max(0, 
        horas_estudio * 0.3 + asistencia * 0.03 + random.gauss(1, 1.5)
    )), 1)
    
    # Nota final: combinación de factores
    nota_final = round(min(10, max(0,
        nota_parcial * 0.4 + horas_estudio * 0.2 + asistencia * 0.02 + random.gauss(0.5, 1)
    )), 1)
    
    # Introducir algunos valores faltantes (realista)
    if random.random() < 0.05:  # 5% de NaN en nota_parcial
        nota_parcial = None
    if random.random() < 0.03:  # 3% de NaN en asistencia
        asistencia = None
    
    datos.append({
        "nombre": nombre,
        "edad": edad,
        "carrera": carrera,
        "horas_estudio": horas_estudio,
        "nota_parcial": nota_parcial,
        "asistencia_pct": asistencia,
        "trabaja": trabaja,
        "nota_final": nota_final,
    })

# Crear DataFrame
df = pd.DataFrame(datos)

print(f"✅ Dataset generado: {len(df)} estudiantes, {len(df.columns)} columnas")
print(f"\n📋 Columnas: {list(df.columns)}")

---
## 4. Cargar y mirar los datos con pandas

Pandas es la biblioteca estándar en Python para trabajar con datos tabulares.
Veamos los comandos esenciales.

In [ ]:
# Ver las primeras 5 filas — SIEMPRE empezá por acá
df.head()

In [ ]:
# Ver las últimas 5 filas
df.tail()

In [ ]:
# Información general: tipos de datos, cantidad de valores no nulos
df.info()

In [ ]:
# Dimensiones del dataset
filas, columnas = df.shape
print(f"📏 Dimensiones: {filas} filas x {columnas} columnas")
print(f"   → {filas} estudiantes con {columnas} atributos cada uno")

In [ ]:
# Tipos de datos de cada columna
print("🔍 Tipos de datos:\n")
for col in df.columns:
    tipo = df[col].dtype
    ejemplo = df[col].dropna().iloc[0]
    print(f"  {col:<20} {str(tipo):<12} ej: {ejemplo}")

In [ ]:
# Acceder a una columna específica
print("📊 Las primeras 10 notas finales:")
print(df["nota_final"].head(10).to_list())

print("\n📊 Carreras únicas:")
print(df["carrera"].unique())

print("\n📊 ¿Cuántos estudiantes por carrera?")
print(df["carrera"].value_counts())

In [ ]:
# Filtrar filas: estudiantes de Sistemas que trabajan
filtro = df[(df["carrera"] == "Sistemas") & (df["trabaja"] == True)]

print(f"👨‍💻 Estudiantes de Sistemas que trabajan: {len(filtro)}")
print(f"   Nota final promedio: {filtro['nota_final'].mean():.2f}")
filtro.head()

---
## 5. Las 3 preguntas obligatorias

Cada vez que recibas un dataset nuevo, respondé estas 3 preguntas antes de hacer CUALQUIER otra cosa:

### Pregunta 1: ¿Cuántos datos tengo?
### Pregunta 2: ¿Hay valores faltantes?
### Pregunta 3: ¿Hay algo raro?

In [ ]:
# ── Pregunta 1: ¿Cuántos datos tengo? ────────────────────────

print("═" * 50)
print("  PREGUNTA 1: ¿Cuántos datos tengo?")
print("═" * 50)
print(f"\n  📏 Filas (estudiantes):  {df.shape[0]}")
print(f"  📏 Columnas (variables): {df.shape[1]}")
print(f"  📏 Celdas totales:       {df.shape[0] * df.shape[1]}")

# Reglas de pulgar para ML
print("\n  📌 Reglas de pulgar:")
print("     • ML clásico: mínimo ~100 filas por categoría")
print("     • Deep Learning: mínimo ~1000-10000 filas")
print("     • LLMs: millones a billones de tokens")
print(f"\n  → Con {df.shape[0]} filas estamos bien para ML clásico ✅")

In [ ]:
# ── Pregunta 2: ¿Hay valores faltantes? ──────────────────────

print("═" * 50)
print("  PREGUNTA 2: ¿Hay valores faltantes (NaN)?")
print("═" * 50)

nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(1)

print(f"\n{'Columna':<20} {'NaN':<8} {'% del total'}")
print("─" * 45)
for col in df.columns:
    n = nulos[col]
    p = pct_nulos[col]
    emoji = "⚠️" if n > 0 else "✅"
    print(f"{emoji} {col:<18} {n:<8} {p}%")

total_nulos = nulos.sum()
print(f"\n📊 Total de valores faltantes: {total_nulos}")
if total_nulos > 0:
    print("   → Hay NaN que vamos a tener que resolver. Lo haremos en la Clase 4.")
else:
    print("   → ¡Dataset limpio! No hay NaN.")

In [ ]:
# ── Pregunta 3: ¿Hay algo raro? ─────────────────────────────

print("═" * 50)
print("  PREGUNTA 3: ¿Hay algo raro?")
print("═" * 50)

# Verificar rangos de cada variable numérica
cols_num = df.select_dtypes(include=["int64", "float64"]).columns

print(f"\n{'Columna':<20} {'Min':<10} {'Max':<10} {'¿Rango OK?'}")
print("─" * 55)

# Rangos esperados
rangos_esperados = {
    "edad":           (18, 35),
    "horas_estudio":  (0, 30),
    "nota_parcial":   (0, 10),
    "asistencia_pct": (0, 100),
    "nota_final":     (0, 10),
}

for col in cols_num:
    vmin = df[col].min()
    vmax = df[col].max()
    if col in rangos_esperados:
        rmin, rmax = rangos_esperados[col]
        ok = vmin >= rmin and vmax <= rmax
        emoji = "✅" if ok else "⚠️ FUERA DE RANGO"
    else:
        emoji = "🔍 verificar"
    print(f"{col:<20} {vmin:<10} {vmax:<10} {emoji}")

# Duplicados
duplicados = df.duplicated().sum()
print(f"\n📊 Filas duplicadas: {duplicados}")

print("\n📌 Si hay valores fuera de rango o duplicados, hay que investigar antes de modelar.")

---
## 6. Estadísticas descriptivas

Las estadísticas descriptivas nos dan un resumen numérico de los datos:

| Métrica | Símbolo | ¿Qué mide? |
|---|---|---|
| **Media** | x̄ | El promedio de todos los valores |
| **Mediana** | Q2 | El valor del medio cuando ordenas todo |
| **Desviación estándar** | σ | Qué tan dispersos están los valores |
| **Mínimo / Máximo** | min/max | Los extremos |
| **Percentiles** | Q1, Q3 | El 25% y 75% de los datos |

> 💡 **Media vs Mediana:** si hay valores extremos (outliers), la mediana es más representativa que la media.

In [ ]:
# El comando mágico: .describe()
df.describe().round(2)

In [ ]:
# Análisis detallado de cada variable numérica

for col in ["edad", "horas_estudio", "nota_parcial", "asistencia_pct", "nota_final"]:
    serie = df[col].dropna()
    print(f"\n{'─' * 50}")
    print(f"📊 {col}")
    print(f"{'─' * 50}")
    print(f"  Media:     {serie.mean():.2f}")
    print(f"  Mediana:   {serie.median():.2f}")
    print(f"  Desv. std: {serie.std():.2f}")
    print(f"  Mín:       {serie.min():.2f}")
    print(f"  Máx:       {serie.max():.2f}")
    
    # ¿Media y mediana están cerca? Si no, hay sesgo
    diff = abs(serie.mean() - serie.median())
    if diff > serie.std() * 0.3:
        print(f"  ⚠️  Media y mediana difieren por {diff:.2f} → posible sesgo en la distribución")
    else:
        print(f"  ✅ Distribución relativamente simétrica")

In [ ]:
# Estadísticas por grupo: ¿cómo varía la nota según carrera y si trabaja?

print("📊 Nota final promedio por CARRERA:\n")
por_carrera = df.groupby("carrera")["nota_final"].agg(["mean", "std", "count"]).round(2)
por_carrera.columns = ["Media", "Desv. Std", "Cantidad"]
print(por_carrera)

print("\n📊 Nota final promedio según si TRABAJA:\n")
por_trabaja = df.groupby("trabaja")["nota_final"].agg(["mean", "std", "count"]).round(2)
por_trabaja.columns = ["Media", "Desv. Std", "Cantidad"]
por_trabaja.index = ["No trabaja", "Trabaja"]
print(por_trabaja)

print("\n💡 ¿Los que trabajan sacan notas más bajas? Miremos las diferencias.")

---
## 7. Visualización de distribuciones

Un gráfico vale más que 100 estadísticas. Veamos cómo se distribuyen los datos.

In [ ]:
# Histogramas de todas las variables numéricas

cols_plot = ["edad", "horas_estudio", "nota_parcial", "asistencia_pct", "nota_final"]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

colores = ["#ff6b6b", "#ffd93d", "#6bcb77", "#4d96ff", "#9b59b6"]

for i, col in enumerate(cols_plot):
    ax = axes[i]
    data = df[col].dropna()
    
    ax.hist(data, bins=20, color=colores[i], edgecolor="#333",
            alpha=0.8, linewidth=0.8)
    
    # Líneas de media y mediana
    ax.axvline(data.mean(), color="red", linestyle="--", linewidth=1.5,
               label=f"Media: {data.mean():.1f}")
    ax.axvline(data.median(), color="blue", linestyle=":", linewidth=1.5,
               label=f"Mediana: {data.median():.1f}")
    
    ax.set_title(col, fontsize=12, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

# Ocultar el sexto subplot vacío
axes[5].axis("off")

fig.suptitle("Distribución de cada variable numérica",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("📊 Línea roja = media | Línea azul = mediana")
print("   Si están muy separadas → la distribución tiene sesgo.")

In [ ]:
# Boxplots: nota_final por carrera y por si trabaja

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Por carrera
carreras_unicas = sorted(df["carrera"].unique())
datos_por_carrera = [df[df["carrera"] == c]["nota_final"].dropna() for c in carreras_unicas]

bp1 = ax1.boxplot(datos_por_carrera, labels=carreras_unicas, patch_artist=True)
for patch, color in zip(bp1["boxes"], ["#ff6b6b", "#ffd93d", "#6bcb77", "#4d96ff"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.set_title("Nota final por carrera", fontsize=12, fontweight="bold")
ax1.set_ylabel("Nota final")
ax1.grid(True, alpha=0.2)

# Por si trabaja
datos_por_trabajo = [
    df[df["trabaja"] == False]["nota_final"].dropna(),
    df[df["trabaja"] == True]["nota_final"].dropna(),
]

bp2 = ax2.boxplot(datos_por_trabajo, labels=["No trabaja", "Trabaja"], patch_artist=True)
for patch, color in zip(bp2["boxes"], ["#6bcb77", "#ff6b6b"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_title("Nota final según si trabaja", fontsize=12, fontweight="bold")
ax2.set_ylabel("Nota final")
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print("📊 El boxplot muestra: mediana (línea central), Q1-Q3 (caja), y outliers (puntos).")
print("   Es la mejor forma de comparar grupos rápidamente.")

In [ ]:
# Scatter plots: relaciones entre variables

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

relaciones = [
    ("horas_estudio", "nota_final", "Horas estudio vs Nota final"),
    ("asistencia_pct", "nota_final", "Asistencia vs Nota final"),
    ("nota_parcial", "nota_final", "Nota parcial vs Nota final"),
]

for ax, (x_col, y_col, titulo) in zip(axes, relaciones):
    # Filtrar NaN para este par
    mask = df[[x_col, y_col]].notna().all(axis=1)
    x = df.loc[mask, x_col]
    y = df.loc[mask, y_col]
    
    ax.scatter(x, y, alpha=0.5, c="#3498db", edgecolors="#333", linewidths=0.3, s=40)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(titulo, fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.2)
    
    # Correlación
    corr = x.corr(y)
    ax.text(0.05, 0.95, f"r = {corr:.2f}", transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

plt.tight_layout()
plt.show()

print("📊 r = correlación: 1.0 = perfecta, 0.0 = ninguna, -1.0 = inversa")
print("   Valores |r| > 0.5 indican relación fuerte.")

---
## 8. Ejercicio evaluable

### Consigna

Respondé las 3 preguntas obligatorias y completá el análisis de una nueva variable.

### Criterio de aprobación

- ✅ Respondiste las 3 preguntas con datos correctos
- ✅ Calculaste estadísticas descriptivas de la variable elegida
- ✅ Creaste al menos 1 gráfico
- ✅ Escribiste una conclusión en tus propias palabras

In [ ]:
# ══════════════════════════════════════════════════════════════
#  EJERCICIO EVALUABLE — Exploración de datos
# ══════════════════════════════════════════════════════════════

# ── Parte 1: responder las 3 preguntas ──────────────────────

mis_respuestas = {
    # Pregunta 1: ¿Cuántos datos tengo?
    "filas":     0,   # TODO: reemplazar con el valor correcto
    "columnas":  0,   # TODO: reemplazar con el valor correcto

    # Pregunta 2: ¿Hay valores faltantes?
    "total_nan":         0,     # TODO: total de NaN en todo el dataset
    "columna_mas_nan":   "",    # TODO: nombre de la columna con más NaN

    # Pregunta 3: ¿Hay algo raro?
    "valores_fuera_rango": "",  # TODO: "sí" o "no" + explicación
    "filas_duplicadas":    0,   # TODO: cantidad de filas duplicadas
}

# ── Validación ──────────────────────────────────────────
checks = []
checks.append(("filas", mis_respuestas["filas"] == df.shape[0]))
checks.append(("columnas", mis_respuestas["columnas"] == df.shape[1]))
checks.append(("total_nan", mis_respuestas["total_nan"] == df.isnull().sum().sum()))

col_mas_nan = df.isnull().sum().idxmax() if df.isnull().sum().sum() > 0 else "ninguna"
checks.append(("columna_mas_nan", mis_respuestas["columna_mas_nan"] == col_mas_nan))
checks.append(("duplicadas", mis_respuestas["filas_duplicadas"] == df.duplicated().sum()))

print("📋 Verificación de respuestas:\n")
aciertos = 0
for nombre, ok in checks:
    emoji = "✅" if ok else "❌"
    print(f"  {emoji} {nombre}")
    if ok:
        aciertos += 1

print(f"\n  Resultado: {aciertos}/{len(checks)} correctas")
if aciertos == len(checks):
    print("  🎉 ¡Todas correctas!")
else:
    print("  📌 Revisá los valores y volvé a ejecutar.")

In [ ]:
# ── Parte 2: análisis exploratorio libre ────────────────────
#
# Elegí UNA variable del dataset y hacé:
# 1. Calcular media, mediana y desviación estándar
# 2. Crear un histograma O boxplot
# 3. Escribir 2-3 oraciones con tu conclusión
#
# TODO: escribí tu código abajo

# Variable elegida:
mi_variable = ""  # TODO: reemplazar con el nombre de la columna

# Tu código de análisis:



# Tu conclusión (como comentario):
# TODO: escribí 2-3 oraciones sobre lo que observaste
#

---
## Resumen de la clase

| Concepto | Lo que aprendiste |
|---|---|
| DataFrame | Tabla de datos en pandas: filas = ejemplos, columnas = variables |
| head / info / describe | Los 3 comandos esenciales para explorar un dataset |
| 3 preguntas | ¿Cuántos datos? ¿Hay NaN? ¿Hay algo raro? |
| Estadísticas | Media, mediana, desviación estándar, percentiles |
| Visualización | Histogramas, boxplots y scatter plots |
| Correlación | r mide la relación lineal entre dos variables |

### Próxima clase

En la **Clase 4** vamos a **limpiar los datos**: resolver los NaN, detectar outliers, y preparar el dataset para modelar.

> 📌 **Recordá:** antes de modelar, siempre hay que MIRAR los datos. `df.head()` es tu mejor amigo.